> **Disclaimer:** This notebook is for *educational and research training purposes only*. It does not constitute medical, clinical, or diagnostic advice. All computational predictions (pLDDT, RMSD, affinity scores) are approximations from machine learning models and must not be treated as experimental results. See the full [DISCLAIMER](https://github.com/JobAiReady/lagos-bio-design/blob/main/DISCLAIMER.md) and [PRIVACY POLICY](https://github.com/JobAiReady/lagos-bio-design/blob/main/PRIVACY_POLICY.md).

# Module 4: Solving African Challenges — Capstone Project
## Lab: Design a Diagnostic Binder for Lassa Virus Glycoprotein

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Apply your skills to design a potential diagnostic binder for Lassa Virus Glycoprotein (GPC).

### Prerequisites
- Completed Modules 1–3
- **GPU Runtime enabled** (Runtime → Change runtime type → T4 GPU)

### Deliverable
The top 3 candidate sequences ready for DNA synthesis ordering.

### Context
Lassa fever affects ~300,000 people annually in West Africa. Current diagnostics are slow and expensive. A computationally designed binder protein could form the basis of a rapid, affordable diagnostic test — similar to how nanobodies are used in COVID rapid tests.

> ⚠️ **GPU Required.** Go to Runtime → Change runtime type → T4 GPU.

---
## Background & Key Concepts

**Lassa fever** kills an estimated 5,000 people annually in West Africa, with Nigeria bearing the heaviest burden. The virus enters human cells by binding its surface **glycoprotein complex (GPC)** to host receptors. This GPC is the primary target for diagnostic binders and therapeutic antibodies — and it's what you'll design against in this capstone.

### The Design-Build-Test-Learn (DBTL) Loop

The engine of modern bioengineering is iterative:

```
Design (RFDiffusion + ProteinMPNN)
  → Build (DNA synthesis, ~$0.07/base)
    → Test (binding assays, SPR, ELISA)
      → Learn (feed results back into next round)
```

Current state-of-the-art achieves a **19% experimental success rate** for computationally designed binders — revolutionary compared to the <0.1% hit rate of traditional screening.

### Epitope Selection

**Epitope mapping** is the critical first step: identifying which region of the GPC surface is:
- **Conserved** across Lassa virus lineages (so your binder works against all strains)
- **Accessible** (not buried inside the protein)
- **Functionally important** (disrupting this site has diagnostic or therapeutic value)

### Key Terms

| Term | Definition |
|------|-----------|
| **GPC (Glycoprotein Complex)** | The surface protein of Lassa virus that mediates entry into human cells. Primary target for diagnostics |
| **Epitope** | The specific region on a target protein that a binder recognizes and attaches to |
| **DBTL Loop** | Design-Build-Test-Learn — the iterative engineering cycle in synthetic biology |
| **Affinity (Kd)** | How strongly a binder attaches to its target. Lower Kd = tighter binding. Nanomolar (nM) range is typical |
| **Conservation** | How unchanged an amino acid position is across strains. Highly conserved = functionally important |
| **pAE (inter-chain)** | When AlphaFold predicts a complex, low pAE between chains suggests the proteins truly interact |

---
## Setup

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU not found! Enable T4 GPU in Runtime settings.'
print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip install -q biopython matplotlib numpy pandas

import os
import numpy as np
import matplotlib.pyplot as plt
from Bio.PDB import PDBParser, PDBIO, Select, NeighborSearch, Superimposer

parser = PDBParser(QUIET=True)

In [ ]:
# Install RFDiffusion & ProteinMPNN if needed
if not os.path.exists('/content/RFdiffusion'):
    !git clone https://github.com/RosettaCommons/RFdiffusion.git /content/RFdiffusion
    %cd /content/RFdiffusion

    # Pin torchdata (datapipes removed in newer versions but DGL needs it)
    !pip install -q torchdata==0.7.1

    # Install DGL (Deep Graph Library) with CUDA support
    !pip install -q dgl -f https://data.dgl.ai/wheels/cu124/repo.html

    # Install vendored SE3-Transformer (removed from PyPI)
    !pip install -q -e env/SE3Transformer

    # Install RFDiffusion + missing transitive deps
    !pip install -q -e .
    !pip install -q hydra-core omegaconf pyrsistent

    os.makedirs('models', exist_ok=True)
    !wget -q -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt
    !wget -q -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt

if not os.path.exists('/content/ProteinMPNN'):
    !git clone https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN

# Install ESMFold for validation
!pip install -q fair-esm
print('All tools ready.')

---
## Step 1: Epitope Mapping — Lassa GPC Analysis
Download the Lassa GPC structure and identify conserved regions suitable for binding.

**PDB ID: 5VK2** — Lassa virus glycoprotein complex (GPC) prefusion structure.

In [ ]:
# Download Lassa GPC structure
import urllib.request
os.makedirs('/content/lassa', exist_ok=True)

urllib.request.urlretrieve(
    'https://files.rcsb.org/download/5VK2.pdb',
    '/content/lassa/5VK2.pdb'
)

# Analyze the structure
structure = parser.get_structure('gpc', '/content/lassa/5VK2.pdb')
model = structure[0]

print('Lassa GPC Structure (5VK2):')
print(f'Chains: {[c.get_id() for c in model.get_chains()]}')
for chain in model:
    n_res = sum(1 for r in chain.get_residues() if r.get_id()[0] == ' ')
    print(f'  Chain {chain.get_id()}: {n_res} residues')

In [ ]:
# Extract the GP1 subunit for targeting
# GP1 is the receptor-binding subunit — ideal for diagnostic binder design
target_chain = 'A'  # Adjust based on 5VK2 chain assignments

class ChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.get_id() == self.chain_id

io = PDBIO()
io.set_structure(structure)
io.save('/content/lassa/gpc_target.pdb', ChainSelect(target_chain))

target = parser.get_structure('target', '/content/lassa/gpc_target.pdb')
n_res = sum(1 for r in target.get_residues() if r.get_id()[0] == ' ')
print(f'Target chain {target_chain} extracted: {n_res} residues')

In [ ]:
# Identify surface-exposed residues (potential epitope)
# Surface residues have high solvent-accessible surface area
from Bio.PDB.DSSP import DSSP

try:
    dssp = DSSP(target[0], '/content/lassa/gpc_target.pdb', dssp='mkdssp')
    surface_residues = []
    for key in dssp.keys():
        rsa = dssp[key][3]  # Relative solvent accessibility
        if rsa > 0.25:  # Surface exposed
            surface_residues.append(key[1][1])
    print(f'Surface-exposed residues: {len(surface_residues)}')
except Exception as e:
    print(f'DSSP not available ({e}). Using B-factor heuristic instead.')
    # Fallback: use residues in a known antigenic region
    surface_residues = list(range(60, 120)) + list(range(180, 240))
    print(f'Using literature-based epitope region: {len(surface_residues)} residues')

# Select hotspot residues for binder design (top surface residues)
hotspot_residues = surface_residues[:10]
print(f'\nSelected hotspot residues for binder design: {hotspot_residues}')

---
## Step 2: Mass Generation of Binder Candidates
Generate multiple candidate binders targeting the conserved epitope.

**Strategy:** Generate many candidates, then screen computationally — mimicking the design-build-test-learn cycle.

In [ ]:
os.chdir('/content/RFdiffusion')
os.makedirs('/content/lassa/candidates', exist_ok=True)

# Format hotspot residues for RFDiffusion
hotspot_str = ','.join([f'{target_chain}{r}' for r in hotspot_residues[:5]])

# Generate 10 binder candidates (use more for a real project: 100-1000)
n_target_res = n_res
!python scripts/run_inference.py \
    inference.input_pdb=/content/lassa/gpc_target.pdb \
    'contigmap.contigs=[{target_chain}1-{n_target_res}/0 60-60]' \
    'ppi.hotspot_res=[{hotspot_str}]' \
    inference.output_prefix=/content/lassa/candidates/lassa_binder \
    inference.model_directory_path=models/ \
    inference.num_designs=10

print('Binder generation complete.')

In [ ]:
# Design sequences for all candidates using ProteinMPNN
import glob

os.chdir('/content/ProteinMPNN')
candidate_pdbs = sorted(glob.glob('/content/lassa/candidates/lassa_binder_*.pdb'))
print(f'Designing sequences for {len(candidate_pdbs)} candidates...')

os.makedirs('/content/lassa/sequences', exist_ok=True)

for pdb in candidate_pdbs:
    name = os.path.basename(pdb).replace('.pdb', '')
    out_dir = f'/content/lassa/sequences/{name}'
    !python protein_mpnn_run.py \
        --pdb_path {pdb} \
        --out_folder {out_dir} \
        --num_seq_per_target 3 \
        --sampling_temp 0.1 \
        --batch_size 1 \
        2>/dev/null

print('Sequence design complete for all candidates.')

---
## Step 3: In-Silico Screening
Validate candidates using ESMFold and filter by pLDDT (>90) and self-consistency RMSD (<2.0 Å).

This is the computational equivalent of experimental screening — we test hundreds of designs computationally before committing to expensive synthesis.

In [ ]:
# Load ESMFold for validation
import esm

esmfold = esm.pretrained.esmfold_v1()
esmfold = esmfold.eval().cuda()
print('ESMFold loaded.')

In [ ]:
# Screen all candidates: fold with ESMFold and compute metrics
import glob
import pandas as pd

results = []

for pdb_path in candidate_pdbs:
    name = os.path.basename(pdb_path).replace('.pdb', '')
    seq_dir = f'/content/lassa/sequences/{name}/seqs'
    fasta_files = glob.glob(f'{seq_dir}/*.fa')
    
    if not fasta_files:
        continue
    
    # Read first designed sequence
    with open(fasta_files[0]) as f:
        lines = f.readlines()
    
    # Get designed sequence (skip native at index 0)
    seq = lines[3].strip() if len(lines) > 3 else lines[1].strip()
    
    try:
        # Fold with ESMFold
        with torch.no_grad():
            pdb_str = esmfold.infer_pdb(seq)
        
        # Save and parse
        folded_path = f'/content/lassa/candidates/{name}_folded.pdb'
        with open(folded_path, 'w') as f:
            f.write(pdb_str)
        
        # Calculate pLDDT
        folded = parser.get_structure('f', folded_path)
        plddt_scores = [a.get_bfactor() for a in folded.get_atoms() if a.get_name() == 'CA']
        mean_plddt = np.mean(plddt_scores)
        
        # Calculate RMSD vs original backbone (binder chain only)
        original = parser.get_structure('o', pdb_path)
        orig_chains = list(original[0].get_chains())
        binder_chain = [c for c in orig_chains if c.get_id() != target_chain]
        
        if binder_chain:
            ca_orig = [a for a in binder_chain[0].get_atoms() if a.get_name() == 'CA']
            ca_fold = [a for a in folded.get_atoms() if a.get_name() == 'CA']
            n = min(len(ca_orig), len(ca_fold))
            
            if n > 10:
                sup = Superimposer()
                sup.set_atoms(ca_orig[:n], ca_fold[:n])
                rmsd = sup.rms
            else:
                rmsd = float('inf')
        else:
            rmsd = float('inf')
        
        results.append({
            'name': name,
            'sequence': seq,
            'length': len(seq),
            'pLDDT': round(mean_plddt, 1),
            'RMSD': round(rmsd, 2),
            'pass': mean_plddt > 80 and rmsd < 2.0
        })
        print(f'{name}: pLDDT={mean_plddt:.1f}, RMSD={rmsd:.2f} Å {"✓" if mean_plddt > 80 and rmsd < 2.0 else "✗"}')
        
    except Exception as e:
        print(f'{name}: Error — {e}')

df = pd.DataFrame(results)
print(f'\nScreened {len(df)} candidates.')

In [ ]:
# Rank and display top candidates
df_sorted = df.sort_values(['pass', 'pLDDT', 'RMSD'], ascending=[False, False, True])

print('\n' + '='*70)
print('TOP 3 CANDIDATES FOR LASSA DIAGNOSTIC BINDER')
print('='*70)

top3 = df_sorted.head(3)
for i, (_, row) in enumerate(top3.iterrows(), 1):
    print(f'\n--- Candidate #{i}: {row["name"]} ---')
    print(f'  pLDDT: {row["pLDDT"]} (threshold: >80)')
    print(f'  RMSD:  {row["RMSD"]} Å (threshold: <2.0)')
    print(f'  Length: {row["length"]} residues')
    print(f'  Sequence: {row["sequence"]}')

print(f'\n\nTotal passing candidates: {df["pass"].sum()} / {len(df)}')

In [ ]:
# Visualization: pLDDT vs RMSD scatter plot
plt.figure(figsize=(8, 6))
colors = ['#22c55e' if p else '#ef4444' for p in df['pass']]
plt.scatter(df['RMSD'], df['pLDDT'], c=colors, s=100, edgecolors='white', linewidths=0.5)
plt.axhline(y=80, color='blue', linestyle='--', alpha=0.3, label='pLDDT > 80')
plt.axvline(x=2.0, color='red', linestyle='--', alpha=0.3, label='RMSD < 2.0 Å')
plt.xlabel('Self-Consistency RMSD (Å)', fontsize=12)
plt.ylabel('Mean pLDDT', fontsize=12)
plt.title('Lassa GPC Binder Screening — Capstone Results', fontsize=14)
plt.legend()

# Annotate top 3
for i, (_, row) in enumerate(top3.iterrows(), 1):
    plt.annotate(f'#{i}', (row['RMSD'], row['pLDDT']),
                 fontsize=10, fontweight='bold', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('/content/lassa/screening_results.png', dpi=150)
plt.show()

In [ ]:
# Export top 3 sequences as FASTA (ready for DNA synthesis ordering)
with open('/content/lassa/top3_candidates.fasta', 'w') as f:
    for i, (_, row) in enumerate(top3.iterrows(), 1):
        f.write(f'>LassaBinder_{i}_{row["name"]}_pLDDT{row["pLDDT"]}_RMSD{row["RMSD"]}\n')
        f.write(f'{row["sequence"]}\n')

print('Top 3 sequences saved to top3_candidates.fasta')
print('These sequences are ready for DNA synthesis ordering.')

# Download results
from google.colab import files
files.download('/content/lassa/top3_candidates.fasta')
files.download('/content/lassa/screening_results.png')

---
## Summary

In this capstone lab you applied the full generative protein design pipeline to a real African health challenge:

1. **Epitope Mapping** — Identified conserved surface regions on Lassa GPC (PDB: 5VK2)
2. **Mass Generation** — Created multiple binder candidates using RFDiffusion
3. **In-Silico Screening** — Validated with ESMFold, filtered by pLDDT (>80) and RMSD (<2.0 Å)
4. **Candidate Selection** — Exported top 3 sequences ready for synthesis

### Real-World Next Steps
- **DNA synthesis** — Order the sequences from a provider like Twist Bioscience or GenScript
- **Expression** — Produce the proteins in E. coli or cell-free systems
- **Binding assays** — Test actual binding to Lassa GPC using SPR or ELISA
- **Diagnostic integration** — If binding confirmed, integrate into lateral flow assay format

This is the methodology used by labs at ACEGID, the Broad Institute, and the Baker Lab.

**Next:** Module 5 — Biosecurity, Ethics & Your Pitch to the Lagos Angel Network